# CVG Harness - Demo Interativo

Este notebook demonstra o uso do CVG Harness para classificar demandas e executar o fluxo operacional.

In [ ]:
# Setup - importar módulos
import sys
sys.path.insert(0, 'src')

from cvg_harness.classification.classifier import classify, save_classification
from cvg_harness.linter.spec_linter import lint_spec
from cvg_harness.guardian.architecture_guardian import ArchitectureGuardian
from cvg_harness.drift.drift_detector import DriftDetector
from cvg_harness.ledger.progress_ledger import ProgressLedger
from cvg_harness.ledger.event_log import Event, EventLog
import tempfile
import json
from pathlib import Path

## 1. Classificação de Demanda

Classificar uma demanda como FAST ou ENTERPRISE.

In [ ]:
# Demanda: ajuste local em UI
dimensions_fast = {
    "impacto_arquitetural": 1,
    "modulos_afetados": 1,
    "risco_de_regressao": 1,
    "criticidade_de_negocio": 0,
    "sensibilidade_de_dados": 0,
    "dependencia_externa": 0,
    "reversibilidade": 1,
    "complexidade_de_validacao": 1,
}

result_fast = classify(
    project="notebook-demo",
    demand="ajuste de CSS em botão",
    dimensions=dimensions_fast,
    rationale="mudança local, baixo impacto"
)

print(f"Modo: {result_fast.mode}")
print(f"Score: {result_fast.total_score}")
print(f"Override: {result_fast.override_applied}")

## 2. Classificação ENTERPRISE

In [ ]:
# Demanda: novo fluxo de autenticação
dimensions_enterprise = {
    "impacto_arquitetural": 3,
    "modulos_afetados": 2,
    "risco_de_regressao": 3,
    "criticidade_de_negocio": 3,
    "sensibilidade_de_dados": 3,
    "dependencia_externa": 2,
    "reversibilidade": 1,
    "complexidade_de_validacao": 2,
}

result_ent = classify(
    project="notebook-demo",
    demand="OAuth2 authentication",
    dimensions=dimensions_enterprise,
    rationale="autenticação é crítica para segurança"
)

print(f"Modo: {result_ent.mode}")
print(f"Score: {result_ent.total_score}")
print(f"Dimensions: {result_ent.dimensions}")

## 3. Spec Linter

Validar qualidade executável da SPEC.

In [ ]:
spec = {
    "version": "v1",
    "criterios": [
        {"descricao": "API retorna 200 OK", "testavel": True},
        {"descricao": "Erro retorna JSON válido", "testavel": True},
    ],
    "modulos": ["auth", "api"],
    "edge_cases": ["token expirado", "rate limit"],
    "limite_escopo": "apenas auth module",
    "areas_proibidas": ["legacy", "v1/deprecated"],
    "contratos": [],
    "fluxo_critico": True,
    "rollback": "reverter migração",
    "observabilidade": "logs de auth",
}

report = lint_spec(spec, mode="ENTERPRISE")
print(f"Resultado: {report.result}")
print(f"Score: {report.score}")
print(f"Falhas bloqueantes: {report.blocking_issues}")
print(f"Warnings: {report.warnings}")

## 4. Architecture Guardian

Verificar aderência arquitetural.

In [ ]:
guardian = ArchitectureGuardian(
    authorized_areas=["src/auth", "src/api"],
    prohibited_areas=["src/legacy", "src/v1"],
)

# Teste com arquivo autorizado
report_ok = guardian.check(["src/auth/login.py"])
print(f"Arquivo autorizado: {report_ok.result}")

# Teste com arquivo proibido
report_fail = guardian.check(["src/legacy/old_login.py"])
print(f"Arquivo proibido: {report_fail.result}")
for v in report_fail.violations:
    print(f"  - {v['rule']}: {v['message']}")

## 5. Drift Detector

Detectar desalinhamento entre camadas.

In [ ]:
detector = DriftDetector(sprint_id="SPRINT-1")

drift_report = detector.detect(
    intake={"problema": "autenticação lenta"},
    prd={"problema": "performance do login"},
    spec={"meta": "otimizar queries"},
)

print(f"Drift: {drift_report.result}")
print(f"Camadas verificadas: {drift_report.layers_checked}")
for f in drift_report.findings:
    print(f"  [{f['severity']}] {f['layer']}: {f['finding']}")

## 6. Progress Ledger

Estado vivo da execução.

In [ ]:
ledger = ProgressLedger.new("notebook-demo", "feature test", "FAST")
ledger.update_gate("GATE_1", "approved")
ledger.update_gate("GATE_2", "approved")
ledger.current_sprint = "SPRINT-1"

print(f"Modo: {ledger.mode}")
print(f"Gate atual: {ledger.current_gate}")
print(f"Sprint: {ledger.current_sprint}")
print(f"Status: {ledger.status}")

## Resumo

O CVG Harness fornece:
- Classificação automática FAST/ENTERPRISE
- Validação de SPEC com regras bloqueantes
- Verificação de aderência arquitetural
- Detecção de drift entre camadas
- Progress tracking e event log
- CLI e REPL interativos